# 📈 યુટ્યુબર સ્ટોક બ્રેકઆઉટ સ્કેનર (SL, T1, T2, T3 સાથે)

## આ કોડ શું કરે છે?
- ✅ 210 NSE સ્ટોક્સ ને સ્કેન કરે છે
- ✅ મૂવિંગ એવરેજ અલાઇનમેન્ટ ચેક કરે છે (30, 50, 200 દિવસ)
- ✅ CAR કonfirmation માંથી ટ્રેન્ડ મજબુત છે કે નહીં તે ચેક કરે છે
- ✅ આપોઆપ SL (સ્ટોપ લૉસ) અને ટાર્ગેટ્સ (T1, T2, T3) નિર્ધારણ કરે છે
- ✅ પરિણામો સરસ ટેબલ ફોર્મેટમાં દર્શાવે છે

## પગલું 1: જરૂરી લાઈબ્રેરી આયાત કરો

આ લાઈબ્રેરીઓ સ્ટોક ડેટા ડાઉનલોડ કરવા, ડેટા પ્રોસેસ કરવા અને પરિણામો દર્શાવવા માટે જરૂરી છે

In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime
import warnings
import logging

# યાહુ ફાઈનાન્સ ચેતવણીઓ કે બિનજરૂરી લૉગ્સ બંધ કરો (સાફ આઉટપુટ માટે)
logging.getLogger('yfinance').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

print('✅ બધી જરૂરી લાઈબ્રેરી લોડ થઈ ગઈ!')

## પગલું 2: મુખ્ય સ્કેનર ફંક્શન બનાવો

આ ફંક્શન:
- સ્ટોક ડેટા ડાઉનલોડ કરે છે
- મૂવિંગ એવરેજ ગણતરી કરે છે (30, 50, 200 દિવસ)
- CAR ચેક કરે છે (ટ્રેન્ડ બળવાન છે કે નહીં)
- SL અને ટાર્ગેટ્સ આપોઆપ ગણતરી કરે છે

In [ ]:
def advanced_stock_scanner(ticker_list):
    """
    સ્ટોક સ્કેનર ફંક્શન - બ્રેકઆઉટ સિગ્નલ શોધે છે
    
    પરિમાણો:
    - ticker_list: NSE સ્ટોક્સની સૂચી (જેમ: ['INFY.NS', 'TCS.NS'])
    
    બધા ફિલ્ટર્સ પાસ કરનાર સ્ટોક્સ વાપસ કરે છે
    """
    
    results = []  # પરિણામો સંગ્રહ કરવા માટે ખાલી તાલિકા
    today_date = datetime.now().strftime('%d-%m-%Y')  # આજનો તારીખ

    print(f'🔍 {len(ticker_list)} સ્ટોક્સ સ્કેન કરી રહ્યા છીએ... કૃપા રાહ જુઓ.\n')

    for ticker in ticker_list:
        try:
            # ===================================
            # પગલું 1: સ્ટોક ડેટા ડાઉનલોડ કરો
            # ===================================
            # 2 વર્ષનો દૈનિક ડેટા ડાઉનલોડ કરો (260 ટ્રેડિંગ દિવસ = 1 વર્ષ)
            data = yf.download(ticker, period='2y', interval='1d', progress=False)

            # ডেટા ચેક કરો - ક્યાં તો ખાલી છે કે બહુ ટૂંકો છે તો આ સ્ટોક છોડી દો
            if data.empty or len(data) < 200:
                continue

            # ===================================
            # પગલું 2: મૂવિંગ એવરેજ ગણતરી કરો
            # ===================================
            close_prices = data['Close'].squeeze()  # બંધ ભાવોને બહાર કાઢો

            # 30, 50, 200 દિવસીય સરેરાશ ગણતરી કરો
            dma_30 = float(close_prices.rolling(window=30).mean().iloc[-1])
            dma_50 = float(close_prices.rolling(window=50).mean().iloc[-1])
            dma_200 = float(close_prices.rolling(window=200).mean().iloc[-1])
            cmp = float(close_prices.iloc[-1])  # આજનો બંધ ભાવ (વર્તમાન ભાવ)

            # 200-DMA થી અંતર ટકા માં ગણતરી કરો
            # આ બતાવે છે કે કિંમત 200-DMA ની કેટલી ઉપર છે
            dist_200_dma = ((cmp - dma_200) / dma_200) * 100

            # ===================================
            # પગલું 3: CAR (Cumulative Average Return) ગણતરી કરો
            # ===================================
            # આ ચેક કરે છે કે ટ્રેન્ડ ક્રમશઃ મજબુત થઈ રહ્યો છે કે નહીં
            
            # છેલ્લા 252 ટ્રેડિંગ દિવસ (1 વર્ષ) માં સર્વોચ્ચ ઉંચાઈ શોધો
            last_1y_data = data.tail(252)
            high_date = last_1y_data['High'].squeeze().idxmax()
            
            # તે તારીખ પછી ની બંધ ભાવો લો
            car_data = close_prices.loc[high_date:]

            # જો ડેટા બહુ ટૂંકો હોય તો આ સ્ટોક છોડી દો
            if len(car_data) < 10:
                continue

            # Cumulative Average Return ગણતરી કરો (વધતી સરેરાશ)
            car_values = car_data.expanding().mean()
            last_10_car = car_values.tail(10)  # છેલ્લા 10 દિવસ

            # આ ચેક કરો કે છેલ્લા 10 દિવસમાં CAR હંમેશા વધી રહ્યો છે
            car_status = 'Positive' if last_10_car.is_monotonic_increasing else 'Negative'

            # ===================================
            # પગલું 4: યુટ્યુબરના ફિલ્ટર્સ લાગુ કરો (બધા 4 શરતો પૂરી થવી જોઈએ)
            # ===================================
            # શરત 1: કિંમત > 30-DMA (ટૂંકા મેદ્દતી ટ્રેન્ડ ઉપર)
            # શરત 2: કિંમત > 50-DMA (મધ્યમ મેદ્દતી ટ્રેન્ડ ઉપર)
            # શરત 3: કિંમત > 200-DMA (લાંબી મેદ્દતી ટ્રેન્ડ ઉપર)
            # શરત 4: CAR હકારાત્મક છે (ટ્રેન્ડ મજબુત થઈ રહ્યો છે)
            
            if not ((cmp > dma_30) and (cmp > dma_50) and (cmp > dma_200) and (car_status == 'Positive')):
                continue  # આ સ્ટોક ફિલ્ટર્સ પાસ કર્યો નહીં, આગલ સ્ટોક પર જાઓ

            # ===================================
            # પગલું 5: SL, ટાર્ગેટ્સ અને રિસ્ક ગણતરી કરો
            # ===================================
            
            # સ્ટોપ લૉસ = છેલ્લા 10 દિવસની સૌથી નીચી કિંમત
            # આ બતાવે છે કે અમે કયા સ્તર પર બહાર નીકળીશું જો ટ્રેડ ખોટો હોય
            swing_low = float(data['Low'].tail(10).min().item())
            sl = swing_low  # સ્ટોપ લૉસ = સ્વિંગ લો
            
            # એન્ટ્રી = વર્તમાન કિંમત
            entry = cmp
            
            # રિસ્ક = એન્ટ્રી - સ્ટોપ લૉસ (રૂપિયામાં કેટલું વધી શકે છે)
            risk = entry - sl

            # જો રિસ્ક અમાન્ય હોય તો આ સ્ટોક છોડી દો
            if risk <= 0:
                continue

            # ટાર્ગેટ્સ ગણતરી કરો (રિસ્ક પર આધારિત)
            # T1 = એન્ટ્રી + (2 × રિસ્ક) - કનિષ્ઠ લક્ષ્ય
            # T2 = એન્ટ્રી + (3 × રિસ્ક) - મુખ્ય લક્ષ્ય
            # T3 = એન્ટ્રી + (5 × રિસ્ક) - આક્રમક લક્ષ્ય
            t1 = entry + (2 * risk)
            t2 = entry + (3 * risk)
            t3 = entry + (5 * risk)

            # ===================================
            # પગલું 6: વધુ માહિતી એકત્રિત કરો
            # ===================================
            
            # 52-સપ્તાહનો શિખર શોધો (છેલ્લા 252 ટ્રેડિંગ દિવસ)
            high_52 = float(data['High'].tail(252).max().item())
            
            # ટ્રેન્ડ શક્તિ ચેક કરો
            # જો 30-DMA > 50-DMA > 200-DMA તો ટ્રેન્ડ મજબુત ઉપવતો છે
            if dma_30 > dma_50 > dma_200:
                trend = 'Strong Uptrend'
            else:
                trend = 'Weak'

            # ===================================
            # પગલું 7: પરિણામો પુષ્ટિ ડેટાબેસમાં ઉમેરો
            # ===================================
            results.append({
                'Date': today_date,
                'Stock': ticker.replace('.NS', ''),
                'CMP': round(entry, 2),
                '30 DMA': round(dma_30, 2),
                '50 DMA': round(dma_50, 2),
                '200 DMA': round(dma_200, 2),
                '200 DMA Dist %': round(dist_200_dma, 2),
                'SL': round(sl, 2),
                'T1': round(t1, 2),
                'T2': round(t2, 2),
                'T3': round(t3, 2),
                'CAR Status': car_status,
                'Action': '🟢 Positive Breakout'
            })

        except Exception as e:
            # જો કોઈ ભૂલ આવે તો આ સ્ટોક છોડી દો અને આગલ સ્ટોક પર જાઓ
            pass

    # ===================================
    # પરિણામો ડેટાફ્રેમમાં રૂપાંતરિત કરો અને બાહ્ય કરો
    # ===================================
    if results:
        df = pd.DataFrame(results)
        # 200 DMA Dist % દ્વારા ક્રમમાં સાજવો (સૌથી વધુ પ્રથમ)
        df = df.sort_values(by='200 DMA Dist %', ascending=True)
        return df
    else:
        return pd.DataFrame()

## પગલું 3: સ્ટોક્સની સૂચી બનાવો અને સ્કેનર ચલાવો

આ 210 મુખ્ય NSE સ્ટોક્સ સ્કેન કરશે

In [ ]:
# 210 NSE સ્ટોક્સની સૂચી
my_stocks = [
    '360ONE.NS', 'ABB.NS', 'APLAPOLLO.NS', 'AUBANK.NS', 'ADANIENSOL.NS',
    'ADANIENT.NS', 'ADANIGREEN.NS', 'ADANIPORTS.NS', 'ADANIPOWER.NS', 'ABCAPITAL.NS',
    'ALKEM.NS', 'AMBER.NS', 'AMBUJACEM.NS', 'ANGELONE.NS', 'APOLLOHOSP.NS',
    'ASHOKLEY.NS', 'ASIANPAINT.NS', 'ASTRAL.NS', 'AUROPHARMA.NS', 'DMART.NS',
    'AXISBANK.NS', 'BSE.NS', 'BAJAJ-AUTO.NS', 'BAJFINANCE.NS', 'BAJAJFINSV.NS',
    'BAJAJHLDNG.NS', 'BANDHANBNK.NS', 'BANKBARODA.NS', 'BANKINDIA.NS', 'BDL.NS',
    'BEL.NS', 'BHARATFORG.NS', 'BHEL.NS', 'BPCL.NS', 'BHARTIARTL.NS',
    'BIOCON.NS', 'BLUESTARCO.NS', 'BOSCHLTD.NS', 'BRITANNIA.NS', 'CGPOWER.NS',
    'CANBK.NS', 'CDSL.NS', 'CHOLAFIN.NS', 'CIPLA.NS', 'COALINDIA.NS',
    'COCHINSHIP.NS', 'COFORGE.NS', 'COLPAL.NS', 'CAMS.NS', 'CONCOR.NS',
    'CROMPTON.NS', 'CUMMINSIND.NS', 'DLF.NS', 'DABUR.NS', 'DALBHARAT.NS',
    'DELHIVERY.NS', 'DIVISLAB.NS', 'DIXON.NS', 'DRREDDY.NS', 'ETERNAL.NS',
    'EICHERMOT.NS', 'EXIDEIND.NS', 'FORCEMOT.NS', 'NYKAA.NS', 'FORTIS.NS',
    'GAIL.NS', 'GVTD.NS', 'GMRAIRPORT.NS', 'GLENMARK.NS', 'GODFRYPHLP.NS',
    'GODREJCP.NS', 'GODREJPROP.NS', 'GRASIM.NS', 'HCLTECH.NS', 'HDFCAMC.NS',
    'HDFCBANK.NS', 'HDFCLIFE.NS', 'HAVELLS.NS', 'HEROMOTOCO.NS', 'HINDALCO.NS',
    'HAL.NS', 'HINDPETRO.NS', 'HINDUNILVR.NS', 'HINDZINC.NS', 'POWERINDIA.NS',
    'HYUNDAI.NS', 'ICICIBANK.NS', 'ICICIGI.NS', 'ICICIPRULI.NS', 'IDFCFIRSTB.NS',
    'ITC.NS', 'INDIANB.NS', 'IEX.NS', 'IOC.NS', 'IRFC.NS', 'IREDA.NS',
    'INDUSTOWER.NS', 'INDUSINDBK.NS', 'NAUKRI.NS', 'INFY.NS', 'INOXWIND.NS',
    'INDIGO.NS', 'JINDALSTEL.NS', 'JSWENERGY.NS', 'JSWSTEEL.NS', 'JIOFIN.NS',
    'JUBLFOOD.NS', 'KEI.NS', 'KPITTECH.NS', 'KALYANIJIL.NS', 'KAYNES.NS',
    'KFINTECH.NS', 'KOTAKBANK.NS', 'LTF.NS', 'LICHSGFIN.NS', 'LTM.NS',
    'LT.NS', 'LAURUSLABS.NS', 'LICI.NS', 'LODHA.NS', 'LUPIN.NS',
    'MM.NS', 'MANAPPURAM.NS', 'MANKIND.NS', 'MARICO.NS', 'MARUTI.NS',
    'MFSL.NS', 'MAXHEALTH.NS', 'MAZDOCK.NS', 'MOTILALOFS.NS', 'MPHASIS.NS',
    'MCX.NS', 'MUTHOOTFIN.NS', 'NBCC.NS', 'NHPC.NS', 'NMDC.NS',
    'NTPC.NS', 'NATIONALUM.NS', 'NESTLEIND.NS', 'NAMINDIA.NS', 'NUVAMA.NS',
    'OBEROIRLTY.NS', 'ONGC.NS', 'OIL.NS', 'PAYTM.NS', 'OFSS.NS',
    'POLICYBZR.NS', 'PGEL.NS', 'PIIND.NS', 'PNBHOUSING.NS', 'PAGEIND.NS',
    'PATANJALI.NS', 'PERSISTENT.NS', 'PETRONET.NS', 'PIDILITIND.NS', 'POLYCAB.NS',
    'PFC.NS', 'POWERGRID.NS', 'PREMIERENE.NS', 'PRESTIGE.NS', 'PNB.NS',
    'RBLBANK.NS', 'RECLTD.NS', 'RADICO.NS', 'RVNL.NS', 'RELIANCE.NS',
    'SBICARD.NS', 'SBILIFE.NS', 'SHREECEM.NS', 'SRF.NS', 'MOTHERSON.NS',
    'SHRIRAMFIN.NS', 'SIEMENS.NS', 'SOLARINDS.NS', 'SONACOMS.NS', 'SBIN.NS',
    'SAIL.NS', 'SUNPHARMA.NS', 'SUPREMEIND.NS', 'SUZLON.NS', 'SWIGGY.NS',
    'TATACONSUM.NS', 'TVSMOTOR.NS', 'TCS.NS', 'TATAELXSI.NS', 'TMPV.NS',
    'TATAPOWER.NS', 'TATASTEEL.NS', 'TECHM.NS', 'FEDERALBNK.NS', 'INDHOTEL.NS',
    'PHOENIXLTD.NS', 'TITAN.NS', 'TORNTPHARM.NS', 'TRENT.NS', 'TIINDIA.NS',
    'UNOMINDA.NS', 'UPL.NS', 'ULTRACEMCO.NS', 'UNIONBANK.NS', 'UNITDSPR.NS',
    'VBL.NS', 'VEDL.NS', 'VMM.NS', 'IDEA.NS', 'VOLTAS.NS',
    'WAAREEENER.NS', 'WIPRO.NS', 'YESBANK.NS', 'ZYDUSLIFE.NS'
]

print('\n' + '='*120)
print('🟢 સ્ટોક સ્કેનર - બ્રેકઆઉટ + SL, T1, T2, T3')
print('='*120 + '\n')

# સ્કેનર ચલાવો અને પરિણામો મેળવો
result = advanced_stock_scanner(my_stocks)

## પગલું 4: પરિણામો દર્શાવો

In [ ]:
# જો કોઈ સ્ટોક ફિલ્ટર્સ પાસ કર્યો હોય તો પરિણામો દર્શાવો
if result.empty:
    print('\n❌ આજે કોઈ સ્ટોક બધી શરતો પૂરી કર્યા નહીં.\n')
else:
    # સૌથી જરૂરી કૉલમ દર્શાવો
    display_cols = ['Date', 'Stock', 'CMP', '30 DMA', '50 DMA', '200 DMA', 
                    '200 DMA Dist %', 'SL', 'T1', 'T2', 'T3', 'CAR Status', 'Action']
    
    print(result[display_cols].to_string(index=False))
    
    print('\n' + '='*120)
    print(f'✅ કુલ સ્ટોક્સ: {len(result)}')
    print('='*120)

## પગલું 5: એક્સેલમાં પરિણામો સાચવો

In [ ]:
# પરિણામોને એક્સેલ ફાઈલમાં નિકાલો જેથી તમે તેનું વિશ્લેષણ કરી શકો
if not result.empty:
    filename = f'Stock_Scanner_Results_{datetime.now().strftime("%d-%m-%Y_%H%M%S")}.xlsx'
    result.to_excel(filename, index=False)
    print(f'✅ પરિણામો સાચવ્યા: {filename}')
else:
    print('❌ એક્સેલમાં સાચવવા માટે કોઈ ડેટા નથી.')

## પગલું 6: પરિણામોનું વર્ણન

### કૉલમ્સ કીર્તન:
- **Date**: સ્કેનનું તારીખ
- **Stock**: શેર કોડ (ઉદા: INFY, TCS)
- **CMP**: વર્તમાન કિંમત (એન્ટ્રી પોઇન્ટ)
- **30 DMA**: 30-દિવસીય ચાલતી સરેરાશ
- **50 DMA**: 50-દિવસીય ચાલતી સરેરાશ
- **200 DMA**: 200-દિવસીય ચાલતી સરેરાશ
- **200 DMA Dist %**: વર્તમાન કિંમત 200-DMA ની કેટલી ઉપર છે (%માં)
- **SL**: સ્ટોપ લૉસ સ્તર (છેલ્લા 10 દિવસ નીચું)
- **T1**: પ્રથમ લક્ષ્ય (2× રિસ્ક)
- **T2**: મુખ્ય લક્ષ્ય (3× રિસ્ક) - અહીં નફો લો
- **T3**: આક્રમક લક્ષ્ય (5× રિસ્ક)
- **CAR Status**: ટ્રેન્ડ મજબુત છે કે નહીં
- **Action**: ખરીદ સંકેત (🟢 = હકારાત્મક)

### ટ્રેડિંગ વર્તમાનપત્ર:
1. **SL પર બહાર નીકળો** જો કિંમત SL સ્તર તોડી જાય
2. **T2 પર આંશિક નફો લો** (અથવા T1 પર)
3. **T3 માં જોખમ ઓછું કરો** (અથવા આક્રમક ટ્રેડર્સ માટે પકડી રાખો)
4. **રીસ્ક:રીવર્ડ ચેક કરો** - સામાન્ય રીતે 1:2 અથવા તેથી વધુ સારું છે